# `get_synonyms()` for All APIs — External Reference

Run from the **project root** so that `scripts/` is on the Python path.

This notebook demonstrates the `get_synonyms()` pipeline for every API client in `scripts/apis_pipe/`. For each API, you will see:
- The raw data returned at each fetch step (printed by the API itself)
- The final compiled synonym DataFrame

**APIs covered:**
- Standard base-class pipeline: COL, GBIF, Index Fungorum, MushroomObserver, GenBank, FishBase
- Custom `get_synonyms()` orchestration: ITIS, Tropicos, Symbiota portals

Each section queries three cases: an **accepted name**, a **known synonym**, and **"Not species"** (a control that should return an empty result).


Notice: If you see an error that says "notebook controller is DISPOSED." when using run all, you can still run the notebook by clicking each run button one at a time. 

Notice: If any changes are made to the codebase, restart the notebook to run with the most up-to-date changes.

In [1]:
import sys, os

# Ensure the project root is on sys.path when running from notebooks/apis_pipe/
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from IPython.display import display

In [ ]:
import time
import pandas as pd

pd.set_option("display.max_colwidth", None)

from scripts.config import (
    COL_PORTAL,
    GBIF_PORTAL,
    GENBANK_PORTAL,
    INDEX_FUNGORUM_PORTAL,
    ITIS_PORTAL,
    MUSHROOM_OBSERVER_PORTAL,
    TROPICOS_PORTAL,
    FISHBASE_PORTAL,
    SYMBIOTA_PORTAL_BY_NAME,
)


def show(df):
    """Display a synonym DataFrame with api_link rendered as a clickable anchor."""
    if df.empty:
        display(df)
        return
    display(
        df.style.format(
            {
                "api_link": lambda url: (
                    f'<a href="{url}" target="_blank">{url}</a>'
                    if url and url not in ("U", "")
                    else url
                )
            }
        )
    )


def run_queries(queries, sleep_between_species=0.0):
    """
    Run get_synonyms() for each (portal, client, accepted_name, synonym_name) tuple.

    For each entry, queries three cases in order:
      1. The accepted name
      2. A known synonym of the accepted name
      3. "Not species" — a control that should return an empty result

    Parameters
    ----------
    queries : list of (portal, client, accepted_name, synonym_name)
    sleep_between_species : float
        Seconds to sleep between each species query. Used for rate-limited APIs.
    """
    for portal, client, accepted, synonym in queries:
        for species, name_type in [
            (accepted, "accepted"),
            (synonym, "synonym"),
            ("Not species", "not species"),
        ]:
            print(f"\n{'='*60}")
            print(f"  {portal.display_name}  ({client.__class__.__name__})")
            print(f"  Species:   {species}")
            if name_type == "synonym":
                print(f"  Name type: {name_type}  (synonym of: {accepted})")
            else:
                print(f"  Name type: {name_type}")
            print(f"{'='*60}")
            try:
                result = client.get_synonyms(species)
                print(f"Synonyms returned: {len(result)}")
                show(result)
            except Exception as e:
                print(f"ERROR: {e}")
            if sleep_between_species:
                time.sleep(sleep_between_species)

: 

: 

---
## How the Standard `get_synonyms()` Pipeline Works

Most API clients in this project inherit `get_synonyms()` directly from the base class (`SpeciesAPI`) without overriding it. The method orchestrates a fixed four-step pipeline:

```python
def get_synonyms(self, name: str) -> pd.DataFrame:
    name = normalize_query_string(name)

    raw_data = self._fetch_query_data(name)
    if self._is_empty(raw_data):
        return empty_synonym_table()

    synonym_data = self._fetch_synonym_data(raw_data)
    accepted_data = self._fetch_accepted_data(raw_data, synonym_data)

    synonyms = self._compile_synonyms(synonym_data) if not self._is_empty(synonym_data) else []
    accepted = self._compile_accepted(accepted_data)  if not self._is_empty(accepted_data) else []

    rows = accepted + synonyms
    return pd.DataFrame(rows) if rows else empty_synonym_table()
```

**Step 1 — `_fetch_query_data(name)`**
Makes the initial API request for the search term. Returns the raw response in the API's native format (JSON dict, list, or XML element). If nothing is found, the pipeline short-circuits and returns an empty DataFrame immediately.

**Step 2 — `_fetch_synonym_data(raw_data)`**
Uses the raw query result to fetch the list of synonyms. For most APIs this means extracting an internal ID from `raw_data` and making a second request to the synonyms endpoint.

**Step 3 — `_fetch_accepted_data(raw_data, synonym_data)`**
Fetches the full record for the accepted name — taxonomy, authorship, publication metadata, etc. Some APIs return the accepted name inline with query results (`raw_data` is reused); others require a separate request.

**Step 4 — `_compile_accepted` and `_compile_synonyms`**
Convert the raw API responses into pipeline-standard row dicts. `_compile_accepted` produces one row for the accepted name (with full taxonomy). `_compile_synonyms` produces one row per synonym. Both call `_format_row` internally to enforce the output schema. The rows are concatenated into the final DataFrame.

**What you'll see printed**
Each `get_synonyms()` call prints the raw API response after each fetch step, labelled `_fetch_query_data`, `_fetch_synonym_data`, and `_fetch_accepted_data`. This is intentional — it lets you inspect exactly what each API returns before any compilation happens.

---
## COL (Catalogue of Life)

COL is a global taxonomic checklist aggregating accepted names and synonymies across all kingdoms, maintained by a consortium of taxonomists. This client queries the ChecklistBank REST API, pinned to the COL26.5 dataset.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [ ]:
from scripts.apis_pipe.col import COLAPI

run_queries([(COL_PORTAL, COLAPI(), "Quercus robur", "Quercus atrosanguinea")])

: 

: 

---
## GBIF (Global Biodiversity Information Facility)

GBIF aggregates occurrence records and taxonomic data contributed by institutions worldwide. This client uses the GBIF backbone taxonomy via `/species/match` to resolve a name to a usage key, then retrieves its synonym list from `/species/{id}/synonyms`.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [ ]:
from scripts.apis_pipe.gbif import GBIFAPI

run_queries([(GBIF_PORTAL, GBIFAPI(), "Amanita muscaria", "Agaricus muscarius")])

: 

: 

---
## Index Fungorum

Index Fungorum is the global nomenclatural database for fungi, maintained by the Royal Botanic Gardens Kew, CAB International, and Landcare Research. It exposes a legacy ASMX web service (XML over HTTP) rather than a REST/JSON API; all responses are parsed as XML. The service can be slow and may time out; requests use a 60-second timeout.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [ ]:
from scripts.apis_pipe.index_fungorum import IndexFungorumAPI

run_queries([(INDEX_FUNGORUM_PORTAL, IndexFungorumAPI(), "Amanita muscaria", "Agaricus muscarius")])

: 

: 

---
## Mushroom Observer

Mushroom Observer is a community-driven database of fungal observations. The JSON API embeds synonym lists directly in each name record, so synonyms require no second network request. Rather than "accepted"/"synonym" labels, Mushroom Observer uses a `deprecated` boolean flag; status is inferred from that field.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [ ]:
from scripts.apis_pipe.mushroomobs import MushroomObserverAPI

run_queries([(MUSHROOM_OBSERVER_PORTAL, MushroomObserverAPI(), "Amanita muscaria", "Amanita amerimuscaria")])

: 

: 

---
## Tropicos

Tropicos is a botanical nomenclature database maintained by the Missouri Botanical Garden, covering vascular plants, bryophytes, algae, and fungi. **Requires a `TROPICOS_API_KEY` in your `.env` file.**

**Custom `get_synonyms()` orchestration.** Unlike the base class, which passes raw data through a fixed pipeline, Tropicos resolves the accepted `NameId` explicitly before fetching synonyms. The fetch sequence is:

1. `_fetch_query_data` — name search
2. `_fetch_accepted_list` — `/Name/{id}/AcceptedNames`, used for ID resolution only
3. `_extract_internal_accepted_id` — reads the accepted NameId (no API call)
4. `_fetch_synonym_data` — `/Name/{accepted_id}/Synonyms`
5. `_fetch_accepted_data` — `/Name/Search` by NameId *(only when the queried name is a synonym; otherwise `raw_data` is reused directly to avoid a redundant fetch)*

This two-step ID resolution is necessary because Tropicos's synonym endpoint requires the *accepted* name's ID, not the queried name's ID — and the queried name may itself be a synonym.

In [ ]:
from scripts.apis_pipe.tropicos import TropicosAPI

run_queries([(TROPICOS_PORTAL, TropicosAPI(), "Quercus robur", "Quercus pedunculata")])

: 

: 

---
## FishBase

FishBase is the world's largest online information system for fish species. It does not expose a public API, so this client scrapes two HTML pages per query: the species summary page (which redirects any name — accepted or synonym — to the accepted species page via a `SpecCode`) and the nomenclature page (which lists all names with status, author, and species code as anchor tag parameters). Subspecific names and misspellings are excluded during compilation.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [ ]:
from scripts.apis_pipe.fishbase import FishBaseAPI

run_queries([(FISHBASE_PORTAL, FishBaseAPI(), "Gadus morhua", "Gadus callarias")])

: 

: 

---
## ITIS (Integrated Taxonomic Information System)

ITIS is an authoritative database of biological names for plants, animals, fungi, and microbes, maintained by a partnership of US federal agencies.

**Custom `get_synonyms()` orchestration.** ITIS exposes data through many small, single-purpose endpoints rather than one compound query. A single lookup requires at least **4 + N** fetch calls, where N is the number of synonyms:

1. `_fetch_query_data` — `getITISTermsFromScientificName` to get the initial TSN
2. `_extract_internal_accepted_id` — calls `_fetch_internal_accepted_id_data` (`getAcceptedNamesFromTSN`) only if the initial result is not already accepted
3. `_fetch_synonym_data` — `getSynonymNamesFromTSN` to get the synonym TSN list, then one `getFullRecordFromTSN` call per synonym
4. `_fetch_accepted_data` — `getFullRecordFromTSN` for the accepted name
5. `_fetch_hierarchy_data` — `getFullHierarchyFromTSN` for the taxonomic hierarchy

Because so many endpoints must be coordinated and each takes the accepted TSN (not raw query data) as input, the orchestration passes the accepted ID explicitly between steps rather than threading raw data through the base-class pipeline. **Expect this section to be noticeably slow.**

In [ ]:
from scripts.apis_pipe.itis import ITISAPI

run_queries([(ITIS_PORTAL, ITISAPI(), "Oncorhynchus mykiss", "Salmo mykiss")])

: 

: 

---
## GenBank (NCBI Entrez)

GenBank is the NIH genetic sequence database maintained by NCBI. Its Taxonomy division provides accepted names and synonym lists via the Entrez E-utilities (`esearch` + `efetch`); all responses are XML.

Uses the **standard base-class** `get_synonyms()` pipeline.

The call here passes `sleep_between_species=1.0`. NCBI rate-limits unauthenticated callers to **3 requests/second**, and each `get_synonyms()` call makes 2 requests (`esearch` + `efetch`), so sleeping 1 second between species queries keeps well within that limit. Without the sleep, rapid back-to-back calls will be throttled or blocked by NCBI. This is not needed in the normal pipeline when used with the frontend.

In [ ]:
from scripts.apis_pipe.genbank import GenBankAPI

run_queries(
    [(GENBANK_PORTAL, GenBankAPI(), "Amanita muscaria", "Agaricus muscarius")],
    sleep_between_species=1.0,
)

: 

: 

---
## Symbiota Portals

Symbiota is open-source biodiversity data management software powering numerous natural history collection portals worldwide. Each portal runs its own instance with independent data but the same API structure. `SymbiotaAPI` is instantiated with the portal name, which resolves the correct base URL from `config.py`.

**Custom `get_synonyms()` orchestration.** Unlike the base class, Symbiota resolves the accepted `tid` explicitly before calling the fetch methods. The fetch sequence is:

1. `_fetch_query_data` — performs two calls: a name search to find the initial `tid`, followed by a full `api/v2/taxonomy/{tid}` fetch; returns a combined taxon record
2. `_extract_internal_accepted_id` — reads the accepted `tid` from the combined record (no API call)
3. `_fetch_synonym_data` — scrapes the `/taxa/index.php` HTML page for the accepted taxon's synonym entries
4. `_fetch_accepted_data` — fetches the full `api/v2/taxonomy/{accepted_tid}` JSON record for taxonomy and authorship *(skipped and `raw_data` reused directly if the queried name is already accepted)*

Symbiota portals do not assign separate IDs to synonym records; a single accepted `tid` is used for both `api_internal_id` and `api_link` across every row.

The portals below all share this implementation and are queried in sequence.

In [ ]:
from scripts.apis_pipe.symbiota import SymbiotaAPI

_SYMBIOTA_QUERIES = [
    # (portal, accepted_species, synonym_of_accepted)
    (SYMBIOTA_PORTAL_BY_NAME["MyCoPortal"],                       "Agaricus campestris",     "Psalliota villatica"),
    (SYMBIOTA_PORTAL_BY_NAME["Lichen Portal"],                    "Xanthoria parietina",     "Physcia parietina"),
    (SYMBIOTA_PORTAL_BY_NAME["Bryophyte Portal"],                 "Pohlia nutans",           "Webera nutans"),
    (SYMBIOTA_PORTAL_BY_NAME["CCH2"],                             "Heteromeles arbutifolia", "Photinia arbutifolia"),
    (SYMBIOTA_PORTAL_BY_NAME["SERNEC"],                           "Magnolia grandiflora",    "Magnolia foetida"),
    (SYMBIOTA_PORTAL_BY_NAME["NANSH"],                            "Rudbeckia hirta",         "Coreopsis hirta"),
    (SYMBIOTA_PORTAL_BY_NAME["Algae Herbarium Portal"],           "Ulva intestinalis",       "Ilea intestinalis"),
    (SYMBIOTA_PORTAL_BY_NAME["Pterido Portal"],                   "Dryopteris filix-mas",    "Nephrodium filix-mas"),
    (SYMBIOTA_PORTAL_BY_NAME["CNH"],                              "Impatiens capensis",      "Impatiens biflora"),
    (SYMBIOTA_PORTAL_BY_NAME["Mid-Atlantic Herbaria Consortium"], "Quercus rubra",           "Quercus maxima"),
    (SYMBIOTA_PORTAL_BY_NAME["swbiodiversity"],                   "Larrea tridentata",       "Larrea mexicana"),
]

run_queries([
    (portal, SymbiotaAPI(portal.display_name), accepted, synonym)
    for portal, accepted, synonym in _SYMBIOTA_QUERIES
])

: 

: 